In [1]:
import sys
print(sys.executable)


c:\Users\Admin\NEIRO\SPR_4_F_2\.venv\Scripts\python.exe


In [1]:
!pip install tqdm


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import sys
from pathlib import Path
print(os.getcwd())

project_root = Path.cwd()  # текущая папка ноутбука
src_path = project_root / "src"
sys.path.append(str(src_path))
print(os.getcwd())

c:\Users\Admin\NEIRO\SPR_4_F_2\src
c:\Users\Admin\NEIRO\SPR_4_F_2\src


In [ ]:
# train_main.py

import os
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from functools import partial
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import pandas as pd
from dataset import avg_ingr_cal, detect_group
# === Пути ===
from pathlib import Path
import sys
import Config as Config

project_root = Path.cwd()  # текущая папка ноутбука
src_path = project_root / "src"
sys.path.append(str(src_path))

# === Импорты из src ===
from dataset import MultimodalDataset, collate_fn, avg_ingr_cal, get_transforms, df_train, df_test, GROUP2ID
from model import MultimodalModel
from Config import Config

# === Тренировка ===
def train():
    import os
    print(os.getcwd())

    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Загружаем CSV
    df_all = pd.read_csv(Config.DATA_CSV)
    df_train = df_all[df_all["split"]=="train"].reset_index(drop=True)
    df_test  = df_all[df_all["split"]=="test"].reset_index(drop=True)

    # Создаём словарь калорийности для известных ингредиентов
    single = df_train[df_train["ingredients"].str.count(";")==0].copy()
    single["cal_per_g"] = single["total_calories"] / single["total_mass"]
    ingr_cal = {r["ingredients"]: r["cal_per_g"] for _, r in single.iterrows()}

    # Вычисляем avg_ingr_cal и cal_per_100g
    for df in [df_train, df_test]:
        df["avg_ingr_cal"] = df["ingredients"].apply(lambda x: avg_ingr_cal(x, ingr_cal))
        df["cal_per_100g"] = df["total_calories"] / df["total_mass"] * 100


    
    # Трансформации
    train_tfm = get_transforms(Config, ds_type="train")
    test_tfm  = get_transforms(Config, ds_type="test")
    
    # Датасеты
    train_ds = MultimodalDataset(df_train, train_tfm)
    test_ds  = MultimodalDataset(df_test, test_tfm)
    
    # DataLoader
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                              collate_fn=partial(collate_fn, tokenizer=train_ds.tokenizer))
    test_loader  = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                              collate_fn=partial(collate_fn, tokenizer=test_ds.tokenizer))
    
    # Модель, оптимизатор, функция потерь
    model = MultimodalModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=Config.HEAD_LR)
    loss_fn = nn.L1Loss()
    
    best_val_mae = float("inf")
    
    for e in range(Config.EPOCHS):
        # ===== TRAIN =====
        model.train()
        train_mae = 0
        for b in tqdm(train_loader, desc=f"Epoch {e+1} [Train]"):
            opt.zero_grad()
            preds = model(b["input_ids"].to(device),
                          b["attention_mask"].to(device),
                          b["image"].to(device),
                          b["avg_cal"].to(device),
                          b["group"].to(device))
            labels = b["label"].to(device)
            loss = loss_fn(preds, labels)
            loss.backward()
            opt.step()
            train_mae += torch.mean(torch.abs(preds - labels)).item()
        train_mae /= len(train_loader)
        
        # ===== TEST =====
        model.eval()
        test_mae = 0
        with torch.no_grad():
            for b in test_loader:
                preds = model(b["input_ids"].to(device),
                              b["attention_mask"].to(device),
                              b["image"].to(device),
                              b["avg_cal"].to(device),
                              b["group"].to(device))
                labels = b["label"].to(device)
                test_mae += torch.mean(torch.abs(preds - labels)).item()
        test_mae /= len(test_loader)
        
        print(f"Epoch {e+1} | Train MAE: {train_mae:.2f} | Test MAE: {test_mae:.2f}")
        
train()


c:\Users\Admin\NEIRO\SPR_4_F_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\Admin\NEIRO\SPR_4_F_2\src


Epoch 1 [Train]:   0%|          | 0/173 [00:00<?, ?it/s]

: 